# Module 3: Managed Tools — Code Interpreter & Browser

![Overview](../shared/img/03.drawio.png)

In this module, you will upgrade Aria with two **AgentCore managed tools** that dramatically expand what she can do.

## What you'll learn

- **AgentCore managed tools** — pre-built, hosted tool services that plug into your agent with a few lines of code
- **Code Interpreter** — sandboxed Python execution for calculations, data analysis, and chart generation
- **Browser Tool** — headless Chrome managed by AgentCore for live web browsing and content extraction

By the end of this module Aria will be able to run Python code and browse the internet — with zero infrastructure for you to manage.

---
## Catch up

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.ensure_ready import ensure_ready

config = ensure_ready("03")

---
## What changed from Module 2 to Module 3?

The Module 2 agent was a plain Strands agent with no tools. The Module 3 agent adds two managed tool integrations. Here is what's new:

### Code Interpreter

- Runs Python in a **sandboxed container** managed by AgentCore
- Supports data analysis, mathematical computation, chart/image generation, and file processing
- The sandbox has **no internet access** — it cannot fetch URLs or call APIs
- Ideal for tasks like "calculate compound interest" or "plot a chart of this data"

### Browser Tool

- **Headless Chrome** managed by AgentCore
- Can navigate to URLs, extract page content, fill forms, and take screenshots
- Gives your agent real-time access to the web
- Ideal for tasks like "look up today's Bitcoin price" or "find the latest blog post"

### Both are managed services

You don't provision instances, manage browser binaries, or handle sandbox security. AgentCore runs the infrastructure and exposes each tool as a simple Python object you pass to your `Agent`.

To see the full agent code, open [agent/main.py](agent/main.py) in a new tab.

---
## Deploy the updated agent

We'll deploy the updated agent to the same runtime. AgentCore handles the versioned rollout.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared import deploy_agent

result = deploy_agent.deploy(
    agent_dir="agent",
    runtime_name="aria_agent",
)

runtime_arn = result["runtime_arn"]
print(f"\nRuntime ARN: {runtime_arn}")

---
## Enable Tracing for Code Interpreter and Browser Tool

Enable **Tracing** on both managed tools so that tool invocations appear in the AgentCore Observability dashboard.

### Code Interpreter

1. Open the **Amazon Bedrock AgentCore** console and navigate to **Built-in tools** > **Code Interpreter**
2. Scroll down to the **Tracing** section and click **Edit**

![Tracing section](../shared/img/tracing-01.png)

3. Toggle **Enable** on and click **Save**

![Enable tracing](../shared/img/tracing-02.png)

### Browser Tool

1. Navigate to **Built-in tools** > **Browser**
2. Scroll down to the **Tracing** section and click **Edit**
3. Toggle **Enable** on and click **Save**

---
## Set up the invocation client

We'll create the boto3 client and a thin local wrapper that keeps the notebook cells clean. The wrapper is defined right here — not hidden in a helper module — so you can see the full `invoke_agent_runtime` call.

In [ ]:
import boto3, json, uuid
import sys; sys.path.insert(0, '..')
from shared import utils

client = boto3.client("bedrock-agentcore", region_name="us-east-1")

def invoke(prompt):
    """Simple wrapper to invoke and stream. Uses a fresh session each time
    to avoid a known issue where the browser tool fails on reuse within
    the same runtime session."""
    session_id = str(uuid.uuid4())
    response = client.invoke_agent_runtime(
        agentRuntimeArn=runtime_arn,
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    return utils.stream_sse_response(response["response"])

---
## Test Code Interpreter

In Module 2, Aria could only answer from her training data. Now she can **write and execute Python code** to give precise, verifiable answers.

### Compound interest calculation

This is a great test because the answer requires real math — not pattern matching.

In [ ]:
# Code Interpreter: sandboxed Python execution
invoke("Calculate compound interest on $10,000 at 7% annually for 30 years")

---
## Test Browser Tool

The browser tool gives Aria access to live information on the web. Let's try a query that requires real-time data.

### Live data lookup

In [ ]:
# Browser Tool: headless Chrome managed by AgentCore
invoke("What is the top headline on Hacker News?")

---
## Go deeper

To learn more about the managed tools used in this module:

- **Code Interpreter:** [AgentCore Code Interpreter documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/code-interpreter-tool.html) — includes file I/O (reading/writing files in the sandbox), supported libraries, and resource limits
- **Browser Tool:** [AgentCore Browser documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-tool.html) — includes browser profiles for persistent cookies/sessions, navigation strategies, and content extraction modes
- **Strands tools integration:** [Strands Agents SDK — Tools](https://github.com/strands-agents/sdk-python) — how managed tools plug into the Strands agent framework

---
## What's next

Aria now has serious capabilities — she can compute, visualize data, and browse the live web. But every conversation still starts from scratch. She has no idea who you are or what you discussed five minutes ago.

In **Module 4: Memory**, we will give Aria persistent memory so she can:
- Remember facts across sessions ("My name is Alice and I work on the payments team")
- Build up context over time
- Deliver a personalized experience that improves with every interaction

---
## Record progress

In [ ]:
import sys; sys.path.insert(0, '..')
from shared import progress

progress.show("03")

---

**Next up: [Module 4 -- Give Aria Persistent Memory](../04-memory/notebook.ipynb)**